In [28]:
import polars as pl
import datetime as dt
from pathlib import Path

In [29]:
pl.Config.set_tbl_cols(200)               
pl.Config.set_tbl_rows(50)                

polars.config.Config

In [30]:
def save_clean_df(df, name, base_dir = "data/clean", fmt = "parquet"):
    """
    Save a cleaned DataFrame in data/clean/.
    Create the directory if it does not exist.
    """

    # Name handling
    if name is None or name == "":
        name = "cleaned_data"

    # Create directory if it does not exist
    base_path = Path(base_dir)
    base_path.mkdir(parents=True, exist_ok=True)

    # Handle existing files
    file_path = base_path / f"{name}.{fmt}"
    i = 1
    while file_path.exists():
        file_path = base_path / f"{name}_{i}.{fmt}"
        i += 1

    # Save DataFrame
    if fmt == "parquet":
        df.write_parquet(file_path)
    elif fmt == "csv":
        df.write_csv(file_path)
    else:
        raise ValueError("Format must be 'parquet' or 'csv'")

    print(f"Saved {name} to {file_path}")
    return

In [31]:
def import_df_trade(path, label = None, save = True, save_name = None, base_dir = "data/clean"):
    """
    Import and clean trade data from a given path.
    """

    origin = dt.datetime(1899, 12, 30)
    label = "" if label is None else f"_{label}"

    # Load data
    df = pl.scan_parquet(path).collect()

    # Clean & rename
    df = df.rename({df.columns[0]: "time"})
    df = df.drop(["trade-stringflag", "trade-rawflag"]).drop_nulls(subset=["time", "trade-price", "trade-volume"])
    df = df.rename({"trade-price": f"price{label}", "trade-volume": f"volume{label}"})

    # Datetime conversion
    df = df.with_columns(pl.col("time").map_elements(lambda x: origin + dt.timedelta(days=float(x))).alias("datetime")).drop("time")
    df = df.with_columns(pl.col("datetime").dt.replace_time_zone("UTC").dt.convert_time_zone("America/New_York"))
    df = df.filter(pl.col("datetime").dt.time().is_between(pl.time(9, 30), pl.time(16, 0))).sort("datetime").select(["datetime", f"price{label}", f"volume{label}"])

    # Aggregate duplicates
    df = df.group_by("datetime").agg([pl.col(f"price{label}").mean().alias(f"price{label}"),pl.col(f"volume{label}").mean().alias(f"volume{label}"),]).sort("datetime")
    
    # Compute log-returns
    df = df.with_columns((pl.col(f"price{label}").log() - pl.col(f"price{label}").log().shift(1)).alias(f"return{label}"))

    # Save cleaned DataFrame
    if save:
        if save_name is None:
            save_name = f"trade{label}"
        save_clean_df(df=df, name=save_name, base_dir=base_dir)

    return df


In [32]:
def import_df_BBO(path, label = None, save = True, save_name = None, base_dir = "data/clean"):
    """
    Import and clean trade data from a given path.
    """

    origin = dt.datetime(1899, 12, 30)
    label = "" if label is None else f"_{label}"

    # Load data
    df = pl.scan_parquet(path).collect()

    # Clean & rename
    df = df.rename({df.columns[0]: "time"})
    df = df.drop_nulls(subset=["bid-price", "bid-volume", "ask-price", "ask-volume"])
    df = df.rename({"bid-price": f"bid_price{label}", "bid-volume": f"bid_volume{label}", "ask-price": f"ask_price{label}", "ask-volume": f"ask_volume{label}"})

    # Datetime conversion
    df = df.with_columns(pl.col("time").map_elements(lambda x: origin + dt.timedelta(days=float(x))).alias("datetime")).drop("time")
    df = df.with_columns(pl.col("datetime").dt.replace_time_zone("UTC").dt.convert_time_zone("America/New_York"))
    df = df.filter(pl.col("datetime").dt.time().is_between(pl.time(9, 30), pl.time(16, 0))).sort("datetime").select(["datetime", f"bid_price{label}", f"bid_volume{label}", f"ask_price{label}", f"ask_volume{label}"])

    # Aggregate duplicates
    df = df.group_by("datetime").agg([pl.col(f"bid_price{label}").mean().alias(f"bid_price{label}"),pl.col(f"bid_volume{label}").mean().alias(f"bid_volume{label}"),pl.col(f"ask_price{label}").mean().alias(f"ask_price{label}"),pl.col(f"ask_volume{label}").mean().alias(f"ask_volume{label}")]).sort("datetime")
    
    # Compute spread
    df = df.with_columns((pl.col(f"ask_price{label}") - pl.col(f"bid_price{label}")).alias(f"spread{label}"))

    # Save cleaned DataFrame
    if save:
        if save_name is None:
            save_name = f"bbo{label}"
        save_clean_df(df=df, name=save_name, base_dir=base_dir)

    return df


In [33]:
def bucket_time_trade(df, freq, price_col = "price", volume_col = "volume", return_col = "return", time_col = "datetime") :
    """
    Aggregate trades into time buckets.
    """
    
    return (
        df
        .group_by_dynamic(
            time_col,
            every=freq,
            period=freq,
            closed="left",
            label="left",
        )
        .agg([
            pl.col(price_col).mean().alias(price_col),
            pl.col(volume_col).sum().alias(volume_col),
            pl.col(return_col).sum().alias(return_col), # Sum of log-returns 
        ])
        .sort(time_col)
    )


In [34]:
def bucket_time_BBO(df, freq, bid_price_col = "bid-price", bid_volume_col = "bid-volume", ask_price_col = "ask-price", ask_volume_col = "ask-volume", spread_col = "spread", time_col = "datetime") :
    """
    Aggregate trades into time buckets.
    """
    
    return (
        df
        .group_by_dynamic(
            time_col,
            every=freq,
            period=freq,
            closed="left",
            label="left",
        )
        .agg([
            pl.col(bid_price_col).mean().alias(bid_price_col),
            pl.col(bid_volume_col).sum().alias(bid_volume_col),
            pl.col(ask_price_col).mean().alias(ask_price_col),
            pl.col(ask_volume_col).sum().alias(ask_volume_col),
            pl.col(spread_col).mean().alias(spread_col),
        ])
        .sort(time_col)
    )


In [35]:
def merging_df(df, assets, save = False, save_name = "merged_data", base_dir = "data/clean"):
    """
    Merge multiple DataFrames on datetime.
    """

    df_merged = df[assets[0]]
    for asset in assets[1:]:
        df_merged = df_merged.join(df[asset], on="datetime", how="inner")

    if save:
        if save_name is None:
            save_name = "merged_data"
        save_clean_df(df=df_merged, name=save_name, base_dir=base_dir)

    return df_merged

def normalzing_df(df):
    """
    Normalize the return columns of a DataFrame.
    """

    for col in df.columns:
        if df[col].dtype == pl.Float64 :
            mean = df[col].mean()
            std = df[col].std()
            df = df.with_columns(((pl.col(col) - mean) / std).alias(f"{col}_normalized"))
    
    return df

In [36]:
base_dir = "../data/clean"
assets = ["AAPL", "SPY"]
df_trade = {}
df_BBO = {}
df_trade_min = {}
df_trade_sec = {}
df_BBO_min = {}
df_BBO_sec = {}
dict_asset_exchange = {"AAPL": "OQ", "SPY": "P"}
save_asset = True
save_merge = True
data_type_vec = ["trade", "BBO"]
dict_data_type_label = { "trade": "", "BBO": ".BBO" }

for data_type in data_type_vec :
    for asset in assets :
        save_name = f"{data_type}_{asset}"
        exchange = dict_asset_exchange[asset]
        data_type_label = dict_data_type_label[data_type]
        path = f"../data/{asset}.{exchange}{data_type_label}/{asset}.{exchange}/*.parquet"

        if data_type == "trade" :
            price_col = f"price_{asset}"
            return_col = f"return_{asset}"
            volume_col = f"volume_{asset}"

            df_trade[asset] = import_df_trade(path=path, label=asset, save=save_asset, save_name=save_name, base_dir=base_dir)

            df_trade_min[asset] = normalzing_df(bucket_time_trade(df=df_trade[asset], freq="1m", price_col=price_col, volume_col=volume_col, return_col=return_col, time_col="datetime"))
            df_trade_sec[asset] = normalzing_df(bucket_time_trade(df=df_trade[asset], freq="1s", price_col=price_col, volume_col=volume_col, return_col=return_col, time_col="datetime"))

        else :
            bid_price_col = f"bid_price_{asset}"
            bid_volume_col = f"bid_volume_{asset}"
            ask_price_col = f"ask_price_{asset}"
            ask_volume_col = f"ask_volume_{asset}"
            spread_col = f"spread_{asset}"

            df_BBO[asset] = import_df_BBO(path=path, label=asset, save=save_asset, save_name=save_name, base_dir=base_dir)

            df_BBO_min[asset] = normalzing_df(bucket_time_BBO(df=df_BBO[asset], freq="1m", bid_price_col=bid_price_col, bid_volume_col=bid_volume_col, ask_price_col=ask_price_col, ask_volume_col=ask_volume_col, spread_col=spread_col, time_col="datetime"))
            df_BBO_sec[asset] = normalzing_df(bucket_time_BBO(df=df_BBO[asset], freq="1s", bid_price_col=bid_price_col, bid_volume_col=bid_volume_col, ask_price_col=ask_price_col, ask_volume_col=ask_volume_col, spread_col=spread_col, time_col="datetime"))

df_trade_min_merged = merging_df(df=df_trade_min, assets=assets, save=save_merge, save_name="merged_1min", base_dir=base_dir)
df_trade_sec_merged = merging_df(df=df_trade_sec, assets=assets, save=save_merge, save_name="merged_1sec", base_dir=base_dir)
df_BBO_min_merged = merging_df(df=df_BBO_min, assets=assets, save=save_merge, save_name="merged_BBO_1min", base_dir=base_dir)
df_BBO_sec_merged = merging_df(df=df_BBO_sec, assets=assets, save=save_merge, save_name="merged_BBO_1sec", base_dir=base_dir)

Saved trade_AAPL to ..\data\clean\trade_AAPL.parquet
Saved trade_SPY to ..\data\clean\trade_SPY.parquet
Saved BBO_AAPL to ..\data\clean\BBO_AAPL.parquet
Saved BBO_SPY to ..\data\clean\BBO_SPY.parquet
Saved merged_1min to ..\data\clean\merged_1min.parquet
Saved merged_1sec to ..\data\clean\merged_1sec.parquet
Saved merged_BBO_1min to ..\data\clean\merged_BBO_1min.parquet
Saved merged_BBO_1sec to ..\data\clean\merged_BBO_1sec.parquet


In [37]:
df_BBO_min_merged.head()

datetime,bid_price_AAPL,bid_volume_AAPL,ask_price_AAPL,ask_volume_AAPL,spread_AAPL,bid_price_AAPL_normalized,bid_volume_AAPL_normalized,ask_price_AAPL_normalized,ask_volume_AAPL_normalized,spread_AAPL_normalized,bid_price_SPY,bid_volume_SPY,ask_price_SPY,ask_volume_SPY,spread_SPY,bid_price_SPY_normalized,bid_volume_SPY_normalized,ask_price_SPY_normalized,ask_volume_SPY_normalized,spread_SPY_normalized
"datetime[μs, America/New_York]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2009-01-02 09:30:00 EST,85.68377,2405.083128,85.810283,8403.712245,0.126513,-1.532667,0.238662,-1.530152,2.374868,5.785224,90.527461,72712.530952,90.542582,63330.188889,0.015122,-0.251416,0.005158,-0.251291,-0.128008,0.205583
2009-01-02 09:31:00 EST,85.528896,2334.266234,85.624263,541.923413,0.095367,-1.536559,0.210451,-1.534826,-0.468177,3.818888,90.532233,63986.966667,90.544984,53010.233333,0.01275,-0.250978,-0.121408,-0.251071,-0.273136,-0.12363
2009-01-02 09:32:00 EST,85.467375,2285.485069,85.577556,1486.348281,0.110182,-1.538105,0.191018,-1.536,-0.126646,4.754197,90.489919,41839.5,90.502652,44751.9,0.012733,-0.254861,-0.442663,-0.254956,-0.389273,-0.126044
2009-01-02 09:33:00 EST,85.901932,1545.955491,85.981782,1715.014813,0.07985,-1.527184,-0.103589,-1.525843,-0.043954,2.839245,90.36483,48585.066667,90.37798,37874.816667,0.01315,-0.26634,-0.344817,-0.266396,-0.485984,-0.068141
2009-01-02 09:34:00 EST,86.386039,2461.548696,86.452374,1462.657468,0.066335,-1.515018,0.261156,-1.514018,-0.135213,1.986041,90.385013,31520.516667,90.398085,42649.5,0.013072,-0.264488,-0.592343,-0.264551,-0.418838,-0.078986
